# Exp7.2.4 — Global-window evaluation

Analysis-only notebook. It reads finalized summary/report CSVs from Exp7.2.4 and never loads checkpoints or per-run artifacts.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'experiment_7_2_4_global_window_eval.py').exists():
            return candidate
    raise FileNotFoundError('Could not find writingRing repo root')

repo_root = find_repo_root()
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_7_2_4_global_window_eval' / 'global_window_eval_v1'
task_only = pd.read_csv(root / 'report_table_task_only.csv')
task_reg = pd.read_csv(root / 'report_table_task_plus_reg.csv')
global_summary = pd.read_csv(root / 'global_window_probe_summary.csv')
deltas = pd.read_csv(root / 'global_vs_valid_delta_summary.csv')
tail = pd.read_csv(root / 'l2_tail_activity_summary.csv')
tail_offsets = pd.read_csv(root / 'l2_tail_offset_summary.csv')


## Task-only — valid length known

This is the original Exp7.2.3 valid-length view, collapsed across the two backbones and three seeds.


In [ ]:
def valid_table(df):
    out = df[['family_label', 'native_ba_valid', 'l2_wc_valid_ba', 'l2_f250_valid_ba', 'valid_phase_gain_pp']].copy()
    for col in ['native_ba_valid', 'l2_wc_valid_ba', 'l2_f250_valid_ba']:
        out[col] = 100.0 * out[col]
    return out.rename(columns={
        'family_label': 'Family',
        'native_ba_valid': 'Native BA (%)',
        'l2_wc_valid_ba': 'L2 WC valid (%)',
        'l2_f250_valid_ba': 'L2 F250 valid (%)',
        'valid_phase_gain_pp': 'F250-WC (pp)',
    }).round(2)

valid_table(task_only)


## Task-only — full 256-step global window

Global WholeCount and Global Fixed250 do not use valid lengths. The native BA column remains the original valid-length native reference.


In [ ]:
def global_table(df):
    out = df[['family_label', 'native_ba_valid', 'l2_wc_global_ba', 'l2_f250_global_ba', 'global_phase_gain_pp', 'wc_global_minus_valid_pp', 'f250_global_minus_valid_pp']].copy()
    for col in ['native_ba_valid', 'l2_wc_global_ba', 'l2_f250_global_ba']:
        out[col] = 100.0 * out[col]
    return out.rename(columns={
        'family_label': 'Family',
        'native_ba_valid': 'Native BA valid ref (%)',
        'l2_wc_global_ba': 'L2 Global WC (%)',
        'l2_f250_global_ba': 'L2 Global F250 (%)',
        'global_phase_gain_pp': 'Global F250-WC (pp)',
        'wc_global_minus_valid_pp': 'WC Global-Valid (pp)',
        'f250_global_minus_valid_pp': 'F250 Global-Valid (pp)',
    }).round(2)

global_table(task_only)


## Task + regularizer — valid length known


In [ ]:
valid_table(task_reg)


## Task + regularizer — full 256-step global window


In [ ]:
global_table(task_reg)


## Architecture-specific endpoint sensitivity

Negative values mean that replacing valid-length aggregation with the full 256-step window reduced test balanced accuracy.


In [ ]:
view = deltas[(deltas['metric'] == 'balanced_accuracy') & deltas['contrast'].isin(['wc_global_minus_valid', 'f250_global_minus_valid'])].copy()
view['delta_pp'] = 100.0 * view['delta_mean']
for regularization in ['task_only', 'task_plus_reg']:
    sub = view[view['regularization'] == regularization]
    pivot = sub.pivot_table(index=['family', 'architecture'], columns='contrast', values='delta_pp')
    ax = pivot.plot(kind='bar', figsize=(12, 5))
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('Global - valid test BA (pp)')
    ax.set_title(f'Endpoint sensitivity — {regularization}')
    plt.tight_layout()
    plt.show()


## L2 post-end tail activity


In [ ]:
test_tail = tail[tail['split'] == 'test'].copy()
for regularization in ['task_only', 'task_plus_reg']:
    sub = test_tail[test_tail['regularization'] == regularization]
    pivot = sub.pivot_table(index='family', columns='architecture', values='tail_fraction_mean')
    ax = (100.0 * pivot).plot(kind='bar', figsize=(11, 5))
    ax.set_ylabel('Tail spikes / all L2 spikes (%)')
    ax.set_title(f'L2 post-end tail fraction — {regularization}')
    plt.tight_layout()
    plt.show()


## Tail decay relative to the true endpoint


In [ ]:
order = ['0_250ms', '250_500ms', '500_750ms', '750_1000ms', 'ge_1000ms']
sub = tail_offsets[(tail_offsets['split'] == 'test') & (tail_offsets['regularization'] == 'task_only')].copy()
sub['offset'] = pd.Categorical(sub['offset'], categories=order, ordered=True)
fig, ax = plt.subplots(figsize=(11, 5))
for family, group in sub.groupby('family', observed=False):
    curve = group.groupby('offset', observed=False)['spikes_per_neuron_second_mean'].mean().reindex(order)
    ax.plot(order, curve.values, marker='o', label=family)
ax.set_ylabel('Tail spikes / neuron / second')
ax.set_xlabel('Offset after endpoint')
ax.set_title('Task-only L2 tail decay')
ax.legend(ncol=2, fontsize=8)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
